# Lab 10 — Attention and the Transformer bridge

**Dataset:** [stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb). First calculate attention directly on tensors, then connect the mechanism to a real pretrained Transformer.

**Failure experiment:** remove the causal mask and measure future-token probability.

## The math behind the experiment

Scaled dot-product attention is

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.
$$

For an autoregressive language model, future positions must be masked. In practice the forbidden score is replaced with $-\infty$ before softmax, so its probability becomes zero.

This is not a cosmetic detail: without the mask, a model predicting the next token can directly inspect information that occurs after the answer it is supposed to predict.

In [ ]:
!pip -q install datasets torch transformers pandas
from datasets import load_dataset
import torch,numpy as np,pandas as pd
D=load_dataset('stanfordnlp/imdb',split='train[:1200]')
Q=torch.tensor([[1.,0.],[0.,1.]])
K=Q.clone(); V=torch.tensor([[10.,0.],[0.,20.]])
scores=Q@K.T/(2**.5); W=torch.softmax(scores,dim=-1)
print('scores\n',scores,'\nweights\n',W,'\noutput\n',W@V)

## 1. Prediction

Attention weights should sum to one along each query row. With a causal mask, token *t* must assign exactly zero probability to future tokens.

In [ ]:
S=torch.tensor([[2.,1.,0.],[2.,2.,1.],[0.,2.,2.]])
mask=torch.triu(torch.ones(3,3),diagonal=1).bool()
causal=S.masked_fill(mask,float('-inf')); causalW=torch.softmax(causal,dim=-1)
leakyW=torch.softmax(S,dim=-1)
print('causal weights\n',causalW)
print('future mass row 1:',causalW[1,2].item())
pd.DataFrame({'position':[0,1,2],'causal_future_mass':causalW[:,2].numpy(),'leaky_future_mass':leakyW[:,2].numpy()})

## 2. Real Transformer bridge

Use a small pretrained encoder classifier. The purpose is to connect the Q/K/V calculation to a production architecture, not to make a new benchmark claim.

In [ ]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification
Tok=AutoTokenizer.from_pretrained('distilbert-base-uncased')
model=AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)
batch=Tok(D['text'][:8],padding=True,truncation=True,return_tensors='pt'); labels=torch.tensor(D['label'][:8])
out=model(**batch,labels=labels)
print('loss',float(out.loss),'logit shape',tuple(out.logits.shape))

## Visualize attention — the weights are a matrix

Each row of an attention matrix corresponds to a query token; each column indicates a key/value position it can read from. In causal attention, entries for future positions are zero after softmax. Heatmaps make this constraint easier to inspect than printed tensors alone.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, matrix, title in [
    (axes[0], W.detach().cpu().numpy(), "Scaled attention weights"),
    (axes[1], causalW.detach().cpu().numpy(), "Causal mask applied"),
    (axes[2], leakyW.detach().cpu().numpy(), "No mask: future leakage")
]:
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_yticks(range(matrix.shape[0]))
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i,j]:.2f}", ha="center", va="center", color="white" if matrix[i,j] < .5 else "black", fontsize=8)
fig.colorbar(im, ax=axes, shrink=.75, label="Attention weight")
fig.tight_layout()
plt.show()

## Conclusion / answer key

The direct calculation should produce normalized attention rows. Causal masking must make future-token probability zero; removing the mask creates an information leak. The pretrained model shows where this primitive sits inside a real Transformer stack.

**Research extension:** compare full attention with local attention at equal sequence length and measure memory, latency, and task quality.